# Phase 7 — encoder_decoder LoRMA prior case retrieval


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm rouge-score


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
import time
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'encoder_decoder_mixed'
MODEL_PATH = 'checkpoints/encoder_decoder_mixed/model.pt'
TOKENIZER_PATH = 'tokenizer_encoder_decoder/tokenizer.json'
MODEL_CONFIG = {'dim': 640, 'encoder_layers': 6, 'decoder_layers': 6, 'n_heads': 16, 'n_kv_heads': 4, 'ff_dim': 1728, 'max_encoder_len': 512, 'max_decoder_len': 256}
MAX_SEQ_LEN = 512  # source encoder budget; decoder has its own budget
USE_GRADIENT_CHECKPOINTING = False
UNFREEZE_LAST_N_BLOCKS = 0  # pure LORMA, as in the supplied notebook
LORMA_LAST_N_BLOCKS = 2  # same attention targets as the supplied notebook
SOURCE_SPANS = 8  # evenly spaced non-overlapping source passages
VAL_MAX_DOCS = 100  # fixed held-out subset used for generation-based selection; None = full dev
LOG_EVERY_DOCS = 1000
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_7_PCR'
WARMUP_RATIO = 0.05
WARMUP_START_FACTOR = 0.1
LR_FACTOR = 0.5
LR_BAD_CHECKS = 2
LR_COOLDOWN_CHECKS = 1
LR_MIN = 1e-5
LR_LOSS_THRESHOLD = 0.001  # require 0.1% relative dev-loss improvement
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_encoder_len']
assert SOURCE_SPANS > 0 and 0 < LORMA_LAST_N_BLOCKS <= min(MODEL_CONFIG['encoder_layers'], MODEL_CONFIG['decoder_layers'])
assert UNFREEZE_LAST_N_BLOCKS == 0  # pure LoRMA
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 5  # retained from the reference; lower this to cap training time
PATIENCE = 10
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(r=16, alpha=32, dropout=0.05, do_pi=True)
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'lorma'
TRAIN_FRACTION = 1.0  # use all training documents after holding out validation

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/pcr_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'pcr_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'pcr_rebuilt_{PHASE1_TAG}_{TECHNIQUE}_len{MAX_SEQ_LEN}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = experiment_metadata('PCR')
    manifest.update(save_tag=SAVE_TAG, best_tag=BEST_TAG, submission_zip=SUBMISSION_ZIP_NAME)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

NEGATIVES_PER_QUERY = 2
TEMPERATURE = 0.07
ENCODE_BATCH_SIZE = 1
IS_ENCODER_DECODER = True
IS_MOE = False

Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/encoder_decoder_mixed/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_encoder_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/pcr_rebuilt_encoder_decoder_mixed_lorma_len512/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/pcr_rebuilt_encoder_decoder_mixed_lorma_len512_best/checkpoint.pt
Submission ZIP: /kaggle/working/pcr_runs/pcr_rebuilt_encoder_decoder_mixed_lorma_len512/pcr_rebuilt_encoder_decoder_mixed_lorma_len512.zip


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID


EOS_ID = EMPTY_ID
BOS_ID = tokenizer.token_to_id('<|bos|>')
DECODER_START_ID = EOS_ID if BOS_ID is None else BOS_ID
PASSAGE_GAP_IDS = tokenizer.encode('\n[...]\n',add_special_tokens=False).ids
SOURCE_BUDGET = MAX_SEQ_LEN
print('Vocabulary:',VOCAB_SIZE,'| case token budget:',SOURCE_BUDGET)

(…)tokenizer_encoder_decoder/tokenizer.json:   0%|          | 0.00/2.39M [00:00<?, ?B/s]

Vocabulary: 33178 | case token budget: 512


In [4]:
def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def case_text(value):
    if isinstance(value,str): return value
    if isinstance(value,list): return '\n'.join(case_text(x) for x in value)
    if isinstance(value,dict):
        for name in ('text','sentence','document'):
            if name in value: return case_text(value[name])
    raise ValueError('Expected case text as a string or a list of sentences')

def normalize_cases(raw,labelled=False):
    if isinstance(raw,list): entries=[(None,r) for r in raw]
    elif isinstance(raw,dict) and 'id' in raw: entries=[(None,raw)]
    elif isinstance(raw,dict): entries=list(raw.items())
    else: raise ValueError('Expected a list or ID-to-record mapping')
    records=[]; seen=set()
    for key,value in entries:
        if isinstance(value,(str,list)) and key is not None: value={'text':value}
        if not isinstance(value,dict): raise ValueError('Each case record must be an object')
        ident=value.get('id',value.get('case_id',value.get('query_id',key)))
        if not isinstance(ident,str): raise ValueError('Expected string case IDs; preserve leading zeros')
        if key is not None and ident!=key: raise ValueError('Case ID does not match mapping key')
        if ident in seen: raise ValueError(f'Duplicate case ID: {ident}')
        seen.add(ident)
        text=case_text(value.get('text',value.get('document'))).strip()
        if not text: raise ValueError(f'{ident}: empty case text')
        record={'id':ident,'text':text}
        if labelled:
            if 'relevant_candidates' not in value:
                raise ValueError(f'{ident}: missing relevant_candidates field; available fields: {list(value)}')
            gold=value['relevant_candidates']
            if not isinstance(gold,list) or any(not isinstance(c,str) for c in gold):
                raise ValueError(f'{ident}: relevant_candidates must be a list of string IDs')
            record['relevant_candidates']=list(dict.fromkeys(gold))
        records.append(record)
    if not records: raise ValueError('No cases found')
    return records

data={}; candidate_pools={}; fingerprints={}
for split,folder in [('train','Train'),('dev','Dev'),('test','Test')]:
    raw_queries=download_json(f'{DATA_PREFIX}/{folder}/{split}.json')
    raw_candidates=download_json(f'{DATA_PREFIX}/{folder}/{split}_candidates.json')
    data[split]=normalize_cases(raw_queries,labelled=split!='test')
    candidate_pools[split]=normalize_cases(raw_candidates)
    pool_ids={r['id'] for r in candidate_pools[split]}
    if len(pool_ids)<7: raise ValueError(f'{split}: fewer than seven candidates')
    for row in data[split]:
        missing=set(row.get('relevant_candidates',[]))-pool_ids
        if missing: raise ValueError(f'{split}/{row["id"]}: relevant IDs absent from this split pool: {sorted(missing)[:5]}')
    digest=hashlib.sha256()
    for records in (data[split],candidate_pools[split]):
        for row in records: digest.update(json.dumps(row,sort_keys=True,ensure_ascii=False).encode())
    fingerprints[split]=digest.hexdigest()
    print(split,'queries:',len(data[split]),'candidates:',len(pool_ids),
          'empty relevance lists:',sum(not r.get('relevant_candidates',[]) for r in data[split]) if split!='test' else 'private')
DATA_FINGERPRINT=fingerprints
train_rows=[r for r in data['train'] if r['relevant_candidates']]
print('Training queries with positives:',len(train_rows),'| skipped empty relevance lists:',len(data['train'])-len(train_rows))
if not train_rows: raise ValueError('No training queries have relevant candidates')
if TRAIN_FRACTION<1:
    train_rows=random.Random(SEED).sample(train_rows,max(1,round(len(train_rows)*TRAIN_FRACTION)))
dev_rows=data['dev'] if VAL_MAX_DOCS is None else data['dev'][:VAL_MAX_DOCS]

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

train queries: 827 candidates: 4320 empty relevance lists: 10


dev.json:   0%|          | 0.00/5.87M [00:00<?, ?B/s]

... (3 outputs left out)

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test queries: 237 candidates: 1727 empty relevance lists: private
Training queries with positives: 817 | skipped empty relevance lists: 10


In [5]:
def select_source_tokens(tokens,budget=SOURCE_BUDGET,spans=SOURCE_SPANS):
    if len(tokens)<=budget: return list(tokens)
    spans=min(spans,budget)
    content_budget=budget-(spans-1)*len(PASSAGE_GAP_IDS)
    if content_budget<spans: raise ValueError('Source budget too small for passage markers')
    widths=[content_budget//spans+int(i<content_budget%spans) for i in range(spans)]
    omitted=len(tokens)-content_budget
    if spans==1: return tokens[:budget]
    selected=[];consumed=0
    for i,width in enumerate(widths):
        start=consumed+i*omitted//(spans-1)
        if i:selected.extend(PASSAGE_GAP_IDS)
        selected.extend(tokens[start:start+width]);consumed+=width
    return selected

query_tokens={};candidate_tokens={}
for split in ('train','dev','test'):
    query_tokens[split]={r['id']:select_source_tokens(tokenizer.encode(r['text'],add_special_tokens=False).ids) or [EOS_ID]
                         for r in tqdm(data[split],desc=f'Tokenize {split} queries')}
    candidate_tokens[split]={r['id']:select_source_tokens(tokenizer.encode(r['text'],add_special_tokens=False).ids) or [EOS_ID]
                             for r in tqdm(candidate_pools[split],desc=f'Tokenize {split} candidates')}

class RetrievalDataset(Dataset):
    def __len__(self):return len(train_rows)
    def __getitem__(self,i):return train_rows[i]

def collate_retrieval(rows):return rows

def make_loader():
    return DataLoader(RetrievalDataset(),batch_size=BATCH_SIZE,shuffle=True,
                      collate_fn=collate_retrieval,num_workers=0)

train_candidate_ids=list(candidate_tokens['train'])
def sample_training_candidates(row):
    positive=random.choice(row['relevant_candidates']);gold=set(row['relevant_candidates'])
    if len(gold)>=len(train_candidate_ids):raise ValueError('Training query has no negative candidates')
    negatives=[]; used=set()
    for _ in range(NEGATIVES_PER_QUERY):
        # Rejection sampling avoids storing a candidate-pool copy for every query.
        for attempt in range(100):
            candidate=random.choice(train_candidate_ids)
            if candidate not in gold and candidate not in used:break
        else:
            eligible=[cid for cid in train_candidate_ids if cid not in gold and cid not in used]
            if not eligible:eligible=[cid for cid in train_candidate_ids if cid not in gold]
            candidate=random.choice(eligible)
        negatives.append(candidate);used.add(candidate)
    return [positive]+negatives

Tokenize train queries:   0%|          | 0/827 [00:00<?, ?it/s]

Tokenize train candidates:   0%|          | 0/4320 [00:00<?, ?it/s]

Tokenize dev queries:   0%|          | 0/118 [00:00<?, ?it/s]

Tokenize dev candidates:   0%|          | 0/1023 [00:00<?, ?it/s]

Tokenize test queries:   0%|          | 0/237 [00:00<?, ?it/s]

Tokenize test candidates:   0%|          | 0/1727 [00:00<?, ?it/s]

In [6]:
RMS_NORM_EPS = 1e-6
ROPE_BASE = 10000
DIM=640
ENCODER_LAYERS=6
DECODER_LAYERS=6
N_HEADS=16
N_KV_HEADS=4
FF_DIM=1728
MAX_ENCODER_LEN=512
MAX_DECODER_LEN=256
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# RMSNorm
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=RMS_NORM_EPS):
        super().__init__()

        self.weight = nn.Parameter(
            torch.ones(dim)
        )

        self.eps = eps

    def forward(self, x):

        rms = (x*x).mean(
            dim=-1,
            keepdim=True
        )

        return self.weight * x * torch.rsqrt(
            rms + self.eps
        )


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):

    def __init__(
        self,
        head_dim,
        max_seq_len,
        base=ROPE_BASE,
    ):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(
                    0,
                    head_dim,
                    2,
                    dtype=torch.float32,
                ) / head_dim
            )
        )

        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        )

        freqs = torch.outer(
            positions,
            inv_freq,
        )

        # [T, head_dim]
        emb = torch.cat(
            [freqs, freqs],
            dim=-1,
        )

        self.register_buffer(
            "cos",
            emb.cos()[None, None, :, :],
            persistent=False,
        )

        self.register_buffer(
            "sin",
            emb.sin()[None, None, :, :],
            persistent=False,
        )

    def forward(self, seq_len):

        return (
            self.cos[:, :, :seq_len],
            self.sin[:, :, :seq_len],
        )


def rotate_half(x):

    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]

    return torch.cat(
        [-x2, x1],
        dim=-1,
    )


def apply_rope(q, k, cos, sin):

    q = (
        q * cos
        +
        rotate_half(q) * sin
    )

    k = (
        k * cos
        +
        rotate_half(k) * sin
    )

    return q, k


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):

    def __init__(
        self,
        dim,
        hidden_dim,
    ):
        super().__init__()

        self.gate_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.up_proj = nn.Linear(
            dim,
            hidden_dim,
            bias=False,
        )

        self.down_proj = nn.Linear(
            hidden_dim,
            dim,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(
                self.gate_proj(x)
            )
            *
            self.up_proj(x)
        )


# ============================================================
# GQA + Flash Attention
# ============================================================

class Attention(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        max_seq_len,
        causal=False,
        use_rope=True,
    ):
        super().__init__()

        assert dim % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.dim = dim
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads

        self.head_dim = dim // n_heads

        self.causal = causal
        self.use_rope = use_rope

        # ----------------------------------------------------
        # Q / K / V projections
        # ----------------------------------------------------

        self.q_proj = nn.Linear(
            dim,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            dim,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.o_proj = nn.Linear(
            n_heads * self.head_dim,
            dim,
            bias=False,
        )

        # RoPE only needs to be constructed for self-attention
        if use_rope:

            self.rope = RotaryEmbedding(
                self.head_dim,
                max_seq_len,
            )

    def forward(
        self,
        x,
        kv=None,
        past_kv=None,
        use_cache=False,
        key_mask=None,
    ):

        B, T, C = x.shape

        # ====================================================
        # Queries
        # ====================================================

        q = self.q_proj(x)

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        )

        q = q.transpose(1, 2)

        # ====================================================
        # Keys / Values
        # ====================================================

        # Self-attention:
        #     source = x
        #
        # Cross-attention:
        #     source = encoder output

        source = x if kv is None else kv

        S = source.shape[1]

        k = self.k_proj(source)
        v = self.v_proj(source)

        k = k.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        v = v.view(
            B,
            S,
            self.n_kv_heads,
            self.head_dim,
        )

        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # ====================================================
        # RoPE
        # ====================================================

        # Only self-attention gets RoPE.
        # Cross-attention does NOT use RoPE.

        if kv is None and self.use_rope:

            current_len = k.shape[2]
            offset = 0 if past_kv is None else past_kv[0].shape[2]
            if offset + current_len > self.rope.cos.size(2):
                raise ValueError('Self-attention exceeds the Phase 1 positional context')
            cos, sin = self.rope(offset + current_len)
            cos = cos[:, :, offset:offset+current_len]
            sin = sin[:, :, offset:offset+current_len]

            q, k = apply_rope(
                q,
                k,
                cos,
                sin,
            )

        # ====================================================
        # Decoder KV Cache
        # ====================================================

        if (
            kv is None
            and past_kv is not None
        ):

            past_k, past_v = past_kv

            k = torch.cat(
                [past_k, k],
                dim=2,
            )

            v = torch.cat(
                [past_v, v],
                dim=2,
            )

        # Store native KV heads before expanding them for GQA.
        cache_kv = (k, v) if use_cache and kv is None else None

        # ====================================================
        # GQA
        # ====================================================

        if self.n_kv_heads != self.n_heads:

            repeat_factor = (
                self.n_heads
                // self.n_kv_heads
            )

            k = k.repeat_interleave(
                repeat_factor,
                dim=1,
            )

            v = v.repeat_interleave(
                repeat_factor,
                dim=1,
            )

        # ====================================================
        # Flash / SDPA Attention
        # ====================================================

        # During cached decoding, only one new token is
        # queried, so causal masking is unnecessary.

        is_causal = (
            self.causal
            and kv is None
            and past_kv is None
        )

        attention_mask = None
        if self.causal and kv is None and past_kv is not None:
            offset = past_kv[0].size(2)
            attention_mask = (torch.arange(k.size(2), device=x.device)[None, :] <=
                              (offset + torch.arange(T, device=x.device))[:, None])[None, None]
        if key_mask is not None:
            valid_keys = key_mask[:, None, None, :].bool()
            attention_mask = valid_keys if attention_mask is None else attention_mask & valid_keys
            if is_causal:
                causal_mask = torch.ones(T, k.size(2), device=x.device, dtype=torch.bool).tril()
                attention_mask = attention_mask & causal_mask[None, None]
            is_causal = False

        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=is_causal,
        )

        # ====================================================
        # Output projection
        # ====================================================

        out = out.transpose(
            1,
            2,
        ).contiguous()

        out = out.view(
            B,
            T,
            C,
        )

        out = self.o_proj(out)

        # Cache only decoder self-attention
        new_kv = None

        if use_cache and kv is None:
            new_kv = cache_kv

        return out, new_kv


# ============================================================
# Encoder Block
# ============================================================

class EncoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_seq_len,
    ):
        super().__init__()

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_seq_len,
            causal=False,
            use_rope=True,
        )

        self.norm2 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(self, x, key_mask=None):

        # Pre-norm self-attention
        x = x + self.self_attn(
            self.norm1(x), key_mask=key_mask
        )[0]

        # Pre-norm FFN
        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_heads,
        ff_dim,
        max_self_attn_len,
        max_cross_attn_len,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Causal self-attention
        # ----------------------------------------------------

        self.norm1 = RMSNorm(dim)

        self.self_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_self_attn_len,
            causal=True,
            use_rope=True,
        )

        # ----------------------------------------------------
        # Encoder-decoder cross-attention
        # ----------------------------------------------------

        self.norm2 = RMSNorm(dim)

        self.cross_attn = Attention(
            dim=dim,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            max_seq_len=max_cross_attn_len,
            causal=False,
            use_rope=False,
        )

        # ----------------------------------------------------
        # FFN
        # ----------------------------------------------------

        self.norm3 = RMSNorm(dim)

        self.ffn = SwiGLU(
            dim,
            ff_dim,
        )

    def forward(
        self,
        x,
        encoder_output,
        past_kv=None,
        use_cache=False,
        key_mask=None,
        encoder_mask=None,
    ):

        # ====================================================
        # 1. Decoder causal self-attention
        # ====================================================

        residual = x

        x_norm = self.norm1(x)

        self_out, new_kv = self.self_attn(
            x_norm,
            past_kv=past_kv,
            use_cache=use_cache,
            key_mask=key_mask,
        )

        x = residual + self_out

        # ====================================================
        # 2. Cross-attention
        # ====================================================

        residual = x

        x_norm = self.norm2(x)

        cross_out, _ = self.cross_attn(
            x_norm,
            kv=encoder_output,
            key_mask=encoder_mask,
        )

        x = residual + cross_out

        # ====================================================
        # 3. FFN
        # ====================================================

        x = x + self.ffn(
            self.norm3(x)
        )

        return x, new_kv


# ============================================================
# Encoder-Decoder Language Model
# ============================================================

class EncoderDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size=VOCAB_SIZE,
        dim=DIM,
        encoder_layers=ENCODER_LAYERS,
        decoder_layers=DECODER_LAYERS,
        n_heads=N_HEADS,
        n_kv_heads=N_KV_HEADS,
        ff_dim=FF_DIM,
        max_encoder_len=MAX_ENCODER_LEN,
        max_decoder_len=MAX_DECODER_LEN,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim

        # ====================================================
        # Shared token embedding
        # ====================================================

        self.token_embedding = nn.Embedding(
            vocab_size,
            dim,
        )

        # ====================================================
        # Encoder
        # ====================================================

        self.encoder_blocks = nn.ModuleList([
            EncoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_seq_len=max_encoder_len,
            )
            for _ in range(encoder_layers)
        ])

        self.encoder_norm = RMSNorm(dim)

        # ====================================================
        # Decoder
        # ====================================================

        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                dim=dim,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                ff_dim=ff_dim,
                max_self_attn_len=max_decoder_len,
                max_cross_attn_len=max_encoder_len,
            )
            for _ in range(decoder_layers)
        ])

        self.decoder_norm = RMSNorm(dim)

        # ====================================================
        # Output head
        # ====================================================

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False,
        )

        # Tie output projection to token embedding
        self.lm_head.weight = (
            self.token_embedding.weight
        )

    # ========================================================
    # ENCODER
    # ========================================================

    def encode(self, encoder_ids):

        x = self.token_embedding(
            encoder_ids
        )

        for block in self.encoder_blocks:
            x = block(x)

        x = self.encoder_norm(x)

        return x

    # ========================================================
    # DECODER
    # ========================================================

    def decode(
        self,
        decoder_ids,
        encoder_output,
        past_kv=None,
        use_cache=False,
    ):

        x = self.token_embedding(
            decoder_ids
        )

        new_cache = []

        for i, block in enumerate(
            self.decoder_blocks
        ):

            layer_past = None

            if past_kv is not None:
                layer_past = past_kv[i]

            x, layer_cache = block(
                x=x,
                encoder_output=encoder_output,
                past_kv=layer_past,
                use_cache=use_cache,
            )

            if use_cache:
                new_cache.append(
                    layer_cache
                )

        x = self.decoder_norm(x)

        logits = self.lm_head(x)

        if use_cache:
            return logits, new_cache

        return logits, None

    # ========================================================
    # FORWARD
    # ========================================================

    def forward(
        self,
        encoder_ids,
        decoder_ids,
        labels=None,
    ):

        # Encode 512-token context
        encoder_output = self.encode(
            encoder_ids
        )

        # Decode continuation
        logits, _ = self.decode(
            decoder_ids,
            encoder_output,
        )

        # No labels -> inference
        if labels is None:
            return logits

        # ====================================================
        # Language-model loss
        # ====================================================

        loss = F.cross_entropy(
            logits.reshape(
                -1,
                self.vocab_size,
            ),
            labels.reshape(-1),
            ignore_index=-100,
        )

        return logits, loss

    # ========================================================
    # GENERATION
    # ========================================================

    @torch.no_grad()
    def generate(
        self,
        encoder_ids,
        bos_id,
        eos_id=None,
        max_new_tokens=128,
        temperature=1.0,
        top_p=0.9,
    ):

        self.eval()

        # ----------------------------------------------------
        # Encode context ONCE
        # ----------------------------------------------------

        encoder_output = self.encode(
            encoder_ids
        )

        # ----------------------------------------------------
        # Start decoder
        # ----------------------------------------------------

        B = encoder_ids.size(0)

        decoder_ids = torch.full(
            (B, 1),
            bos_id,
            dtype=torch.long,
            device=encoder_ids.device,
        )

        past_kv = None

        # ----------------------------------------------------
        # Autoregressive generation
        # ----------------------------------------------------

        for step in range(max_new_tokens):

            # Only feed the newest token once cache exists
            if past_kv is None:
                current_ids = decoder_ids
            else:
                current_ids = decoder_ids[:, -1:]

            logits, past_kv = self.decode(
                decoder_ids=current_ids,
                encoder_output=encoder_output,
                past_kv=past_kv,
                use_cache=True,
            )

            next_logits = logits[:, -1, :]

            # Temperature
            if temperature != 1.0:
                next_logits = (
                    next_logits / temperature
                )

            # ------------------------------------------------
            # Top-p sampling
            # ------------------------------------------------

            probs = F.softmax(
                next_logits,
                dim=-1,
            )

            sorted_probs, sorted_indices = torch.sort(
                probs,
                descending=True,
                dim=-1,
            )

            cumulative_probs = torch.cumsum(
                sorted_probs,
                dim=-1,
            )

            remove = (
                cumulative_probs > top_p
            )

            # Keep at least the first token
            remove[:, 0] = False

            sorted_probs[remove] = 0.0

            sorted_probs = (
                sorted_probs
                /
                sorted_probs.sum(
                    dim=-1,
                    keepdim=True,
                )
            )

            next_token = torch.multinomial(
                sorted_probs,
                num_samples=1,
            )

            next_token = torch.gather(
                sorted_indices,
                -1,
                next_token,
            )

            decoder_ids = torch.cat(
                [
                    decoder_ids,
                    next_token,
                ],
                dim=1,
            )

            if (
                eos_id is not None
                and torch.all(
                    next_token == eos_id
                )
            ):
                break

        # Remove BOS
        return decoder_ids[:, 1:]


# ============================================================
# Parameter count
# ============================================================




In [7]:
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05, do_pi=True):
        super().__init__()
        self.original_linear = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling, self.do_pi = alpha / r, do_pi
        self.dropout = nn.Dropout(dropout)
        for p in original_linear.parameters(): p.requires_grad = False
        W = original_linear.weight
        self.A = nn.Parameter(W.new_empty(r, self.out_features))
        self.B = nn.Parameter(W.new_zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def effective_weight(self):
        W = self.original_linear.weight
        delta = self.B @ self.A
        if self.do_pi:
            rows = torch.arange(self.out_features, device=W.device)[:, None]
            cols = torch.arange(self.out_features, device=W.device)[None, :]
            delta = delta.gather(1, (cols - rows) % self.out_features)
        return W + self.scaling * (delta @ W)
    def forward(self, x):
        W = self.original_linear.weight
        W_eff = self.effective_weight()
        return F.linear(x, W, self.original_linear.bias) + F.linear(self.dropout(x), W_eff - W)


from torch.utils.checkpoint import checkpoint as activation_checkpoint
from contextlib import contextmanager

class RetrievalBackbone(EncoderDecoderLM):
    def encode(self, encoder_ids, encoder_mask=None):
        x = self.token_embedding(encoder_ids)
        for block in self.encoder_blocks:
            if self.training and USE_GRADIENT_CHECKPOINTING:
                x = activation_checkpoint(block, x, key_mask=encoder_mask, use_reentrant=False,
                                          preserve_rng_state=True)
            else: x = block(x, key_mask=encoder_mask)
        return self.encoder_norm(x)

    def decode(self, decoder_ids, encoder_output, past_kv=None, use_cache=False,
               decoder_mask=None, encoder_mask=None):
        x = self.token_embedding(decoder_ids)
        new_cache = []
        for i, block in enumerate(self.decoder_blocks):
            layer_past = None if past_kv is None else past_kv[i]
            if self.training and USE_GRADIENT_CHECKPOINTING:
                if use_cache: raise ValueError('Training checkpointing does not use the inference cache')
                x, layer_cache = activation_checkpoint(block, x, encoder_output,
                    key_mask=decoder_mask, encoder_mask=encoder_mask, use_reentrant=False,
                    preserve_rng_state=True)
            else:
                x, layer_cache = block(x, encoder_output, past_kv=layer_past, use_cache=use_cache,
                                       key_mask=decoder_mask, encoder_mask=encoder_mask)
            if use_cache: new_cache.append(layer_cache)
        return self.lm_head(self.decoder_norm(x)), new_cache if use_cache else None

    def forward(self, encoder_ids, decoder_ids, encoder_mask=None, decoder_mask=None):
        memory = self.encode(encoder_ids, encoder_mask)
        logits, _ = self.decode(decoder_ids, memory, encoder_mask=encoder_mask,
                                decoder_mask=decoder_mask)
        return logits

phase1_path = download_bucket_file(MODEL_PATH)

def build_retrieval_model():
    model = RetrievalBackbone(vocab_size=VOCAB_SIZE, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint: state = checkpoint['model_state_dict']
    elif 'model' in checkpoint: state = checkpoint['model']
    elif 'state_dict' in checkpoint: state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()): state = checkpoint
    else: raise ValueError(f'Cannot identify weights: {list(checkpoint)}')
    # Restore the original vocabulary head too; no Phase 5 classification weights.
    state = dict(state)
    if 'lm_head.weight' not in state and 'token_embedding.weight' in state:
        state['lm_head.weight'] = state['token_embedding.weight']
    if not torch.equal(state['lm_head.weight'], state['token_embedding.weight']):
        raise ValueError('Checkpoint LM head and embedding disagree with the tied Phase 1 architecture')
    model.load_state_dict(state, strict=True)
    del state, checkpoint
    for p in model.parameters(): p.requires_grad = False
    attention_modules = [b.self_attn for b in model.encoder_blocks[-LORMA_LAST_N_BLOCKS:]]
    attention_modules += [a for b in model.decoder_blocks[-LORMA_LAST_N_BLOCKS:]
                          for a in (b.self_attn, b.cross_attn)]
    for attention in attention_modules:
        for name in ['q_proj', 'k_proj', 'v_proj', 'o_proj']:
            setattr(attention, name, LoRMALinear(getattr(attention, name), **ADAPTER_CONFIG))
    model = model.to(DEVICE)
    assert model.lm_head.weight is model.token_embedding.weight
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model



def retrieval_hidden(model,ids,mask):
    if IS_ENCODER_DECODER:
        memory=model.encode(ids,encoder_mask=mask)
        mean=(memory.float()*mask[...,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)
        # A one-token decoder readout uses the pretrained decoder and cross-attention too.
        x=model.token_embedding(torch.full((len(ids),1),DECODER_START_ID,device=ids.device,dtype=torch.long))
        for block in model.decoder_blocks:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x,_=activation_checkpoint(block,x,memory,encoder_mask=mask,use_reentrant=False,preserve_rng_state=True)
            else:x,_=block(x,memory,encoder_mask=mask)
        return (mean+model.decoder_norm(x[:,0]).float())/2
    x=model.token_embedding(ids)
    for block in model.blocks:
        if IS_MOE:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x,_=activation_checkpoint(block,x,valid_mask=mask,use_reentrant=False,preserve_rng_state=True)
            else:x,_=block(x,valid_mask=mask)
        else:
            if model.training and USE_GRADIENT_CHECKPOINTING:
                x=activation_checkpoint(block,x,use_reentrant=False,preserve_rng_state=True)
            else:x=block(x)
    x=model.final_norm(x)
    return (x.float()*mask[...,None]).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)

def embed_sequences(model,sequences):
    vectors=[]
    for start in range(0,len(sequences),ENCODE_BATCH_SIZE):
        part=sequences[start:start+ENCODE_BATCH_SIZE];width=max(map(len,part))
        ids=torch.full((len(part),width),PAD_ID,device=DEVICE,dtype=torch.long)
        mask=torch.zeros_like(ids,dtype=torch.bool)
        for i,seq in enumerate(part):
            ids[i,:len(seq)]=torch.tensor(seq,device=DEVICE);mask[i,:len(seq)]=True
        with autocast_context():v=retrieval_hidden(model,ids,mask)
        vectors.append(F.normalize(v.float(),dim=-1))
    return torch.cat(vectors)

def retrieval_loss_sum(model,rows):
    total=None
    for row in rows:
        candidates=sample_training_candidates(row)
        sequences=[query_tokens['train'][row['id']]]+[candidate_tokens['train'][cid] for cid in candidates]
        embeddings=embed_sequences(model,sequences)
        scores=(embeddings[:1]@embeddings[1:].T)/TEMPERATURE
        loss=F.cross_entropy(scores,torch.zeros(1,dtype=torch.long,device=DEVICE))
        total=loss if total is None else total+loss
    return total,len(rows)

@torch.no_grad()
def rank_cases(model,split,rows=None):
    was_training=model.training;model.eval()
    try:
        if rows is None:rows=data[split]
        ids=list(candidate_tokens[split]);embeddings=[];started=time.perf_counter()
        for start in tqdm(range(0,len(ids),ENCODE_BATCH_SIZE),desc=f'Encode {split} pool'):
            batch=ids[start:start+ENCODE_BATCH_SIZE]
            embeddings.append(embed_sequences(model,[candidate_tokens[split][cid] for cid in batch]).cpu())
        matrix=torch.cat(embeddings);predictions={};hits=gold_count=0
        for row in tqdm(rows,desc=f'Rank {split}'):
            q=embed_sequences(model,[query_tokens[split][row['id']]])[0].cpu()
            # Sorting stable ties follows the candidate file order.
            selected=torch.argsort(matrix@q,descending=True,stable=True)[:7].tolist()
            predictions[row['id']]=[ids[i] for i in selected]
            if split!='test':
                gold=set(row['relevant_candidates']);gold_count+=len(gold)
                hits+=len(gold.intersection(predictions[row['id']]))
        p=hits/(7*len(rows)) if rows else 0.0;r=hits/gold_count if gold_count else 0.0
        metrics=dict(micro_precision_at_7=p,micro_recall_at_7=r,micro_f1_at_7=2*p*r/(p+r) if p+r else 0.0,
                     hits=hits,gold_relevant=gold_count,queries=len(rows),ranking_seconds=time.perf_counter()-started)
        return predictions,metrics
    finally:model.train(was_training)

(…)eckpoints/encoder_decoder_mixed/model.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

In [8]:
def experiment_metadata(domain='PCR'):
    # Epoch target is deliberately not part of compatibility: it may be extended on resume.
    return dict(domain=domain,phase1_tag=PHASE1_TAG,model_path=MODEL_PATH,tokenizer_path=TOKENIZER_PATH,
                model_config=MODEL_CONFIG,adapter_config=ADAPTER_CONFIG,lorma_last_n_blocks=LORMA_LAST_N_BLOCKS,
                source_budget=SOURCE_BUDGET,source_spans=SOURCE_SPANS,data_fingerprint=DATA_FINGERPRINT,
                batch_size=BATCH_SIZE,grad_accum_steps=GRAD_ACCUM_STEPS,lr=LR,weight_decay=WEIGHT_DECAY,
                negatives_per_query=NEGATIVES_PER_QUERY,temperature=TEMPERATURE,seed=SEED,
                train_fraction=TRAIN_FRACTION,val_max_docs=VAL_MAX_DOCS,
                scheduler=dict(warmup_ratio=WARMUP_RATIO,start_factor=WARMUP_START_FACTOR,factor=LR_FACTOR,
                               bad_checks=LR_BAD_CHECKS,cooldown=LR_COOLDOWN_CHECKS,min_lr=LR_MIN,threshold=LR_LOSS_THRESHOLD))

def save_retrieval_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_retrieval_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists():
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    if payload['metadata'] != experiment_metadata(domain):
        changed = [k for k in set(payload['metadata']) | set(experiment_metadata(domain)) if payload['metadata'].get(k) != experiment_metadata(domain).get(k)]
        raise ValueError('Checkpoint configuration mismatch: ' + ', '.join(sorted(changed)))
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload



class ValidationLRScheduler:
    def __init__(self, optimizer, total_updates):
        assert total_updates > 0 and 0 < WARMUP_RATIO < 1
        assert 0 < WARMUP_START_FACTOR <= 1 and 0 < LR_FACTOR < 1
        assert LR_BAD_CHECKS >= 1 and 0 <= LR_MIN <= LR
        self.optimizer = optimizer
        self.updates = 0
        self.warmup_updates = max(1, math.ceil(total_updates * WARMUP_RATIO))
        self.peak_lrs = [g['lr'] for g in optimizer.param_groups]
        # PyTorch reduces after num_bad_epochs > patience: 1 means two bad checks.
        self.plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR, patience=LR_BAD_CHECKS-1,
            threshold=LR_LOSS_THRESHOLD, threshold_mode='rel',
            cooldown=LR_COOLDOWN_CHECKS, min_lr=LR_MIN)
        self._set_warmup_lr()

    def _set_warmup_lr(self):
        factor = WARMUP_START_FACTOR + (1-WARMUP_START_FACTOR) * self.updates/self.warmup_updates
        for group, peak in zip(self.optimizer.param_groups, self.peak_lrs):
            group['lr'] = peak * factor

    def optimizer_step_completed(self):
        self.updates += 1
        if self.updates <= self.warmup_updates: self._set_warmup_lr()

    def validation_step(self, loss):
        if not math.isfinite(loss): raise ValueError('Non-finite validation loss')
        # Warmup and plateau never fight over the learning rate.
        if self.updates >= self.warmup_updates: self.plateau.step(loss)

    def state_dict(self):
        return dict(updates=self.updates, warmup_updates=self.warmup_updates,
                    peak_lrs=self.peak_lrs, plateau=self.plateau.state_dict(),
                    current_lrs=[g['lr'] for g in self.optimizer.param_groups])

    def load_state_dict(self, state):
        self.warmup_updates = state['warmup_updates']  # restore the original horizon when extending epochs
        self.updates = state['updates']
        self.peak_lrs = state['peak_lrs']
        self.plateau.load_state_dict(state['plateau'])
        for group, lr in zip(self.optimizer.param_groups, state['current_lrs']): group['lr'] = lr

In [9]:
def train_retrieval(domain='PCR'):
    assert_run_destination();seed_everything(SEED)
    model=build_retrieval_model()
    params=[p for p in model.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE==torch.float16)
    loader=make_loader();updates_per_epoch=math.ceil(len(loader)/GRAD_ACCUM_STEPS)
    scheduler=ValidationLRScheduler(optimizer,NUM_EPOCHS*updates_per_epoch)
    start=0;extra=dict(best_f1=-1.0,patience_reference=-1.0,stale=0,history=[],best_tag=BEST_TAG)
    if RESUME:
        payload=load_retrieval_checkpoint(SAVE_TAG,model,domain,optimizer,restore_rng=True)
        start=payload['epoch'];extra=payload['extra']
        scaler.load_state_dict(extra['scaler']);scheduler.load_state_dict(extra['scheduler'])
        del payload
        print('Resume at completed epoch:',start,'| target epochs:',NUM_EPOCHS)
    for epoch in range(start,NUM_EPOCHS):
        if extra['stale']>=PATIENCE:break
        model.train();optimizer.zero_grad(set_to_none=True)
        train_sum=train_n=group_n=seen=0
        for step,rows in enumerate(tqdm(loader,desc=f'PCR epoch {epoch+1}')):
            loss_sum,count=retrieval_loss_sum(model,rows)
            if not torch.isfinite(loss_sum):raise FloatingPointError('Nonfinite retrieval loss')
            scaler.scale(loss_sum).backward()
            train_sum+=loss_sum.item();train_n+=count;group_n+=count;seen+=count
            if (step+1)%GRAD_ACCUM_STEPS==0 or step+1==len(loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None:p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params,GRAD_CLIP)
                previous=scaler.get_scale();scaler.step(optimizer);scaler.update()
                if scaler.get_scale()>=previous:scheduler.optimizer_step_completed()
                optimizer.zero_grad(set_to_none=True);group_n=0
            if seen and seen%LOG_EVERY_DOCS<len(rows):print('Queries:',seen,'| loss:',train_sum/train_n)
        predictions,metrics=rank_cases(model,'dev',dev_rows)
        f1=metrics['micro_f1_at_7'];scheduler.validation_step(1.0-f1)
        improved=f1>extra['best_f1']
        if improved:extra['best_f1']=f1;extra['best_metrics']=metrics
        if f1>extra['patience_reference']+MIN_DELTA:extra['patience_reference']=f1;extra['stale']=0
        else:extra['stale']+=1
        extra['history'].append(dict(epoch=epoch+1,train_loss=train_sum/train_n,lr=optimizer.param_groups[0]['lr'],**metrics))
        extra['scaler']=scaler.state_dict();extra['scheduler']=scheduler.state_dict()
        print('Dev retrieval:',metrics)
        if improved:save_retrieval_checkpoint(BEST_TAG,model,optimizer,epoch+1,extra,domain)
        save_retrieval_checkpoint(SAVE_TAG,model,optimizer,epoch+1,extra,domain)
        with open(WORK_DIR/'PCR_history.json','w') as f:json.dump(extra['history'],f,indent=2)
    if not extra['history']:raise ValueError('No trained checkpoint available; NUM_EPOCHS must be positive')
    load_retrieval_checkpoint(BEST_TAG,model,domain)
    with open(WORK_DIR/'PCR_best_dev_metrics.json','w') as f:json.dump(extra['best_metrics'],f,indent=2)
    # Retain latest for exact continuation; it differs from the best validation epoch.
    del optimizer
    return model,extra['history']

In [10]:
def validate_predictions(predictions,documents,pool):
    if set(predictions)!={row['id'] for row in documents}:raise ValueError('Prediction query IDs do not match Test queries')
    allowed={row['id'] for row in pool}
    for qid,ids in predictions.items():
        if len(ids)!=7 or len(set(ids))!=7:raise ValueError(f'{qid}: expected seven unique IDs')
        if not set(ids)<=allowed:raise ValueError(f'{qid}: prediction outside the Test pool')

model,history=train_retrieval()
predictions,_=rank_cases(model,'test')
validate_predictions(predictions,data['test'],candidate_pools['test'])
with open(WORK_DIR/'prediction.json','w',encoding='utf-8') as f:json.dump(predictions,f,ensure_ascii=False,indent=2)

Total/trainable parameters: 79806720 307200


PCR epoch 1:   0%|          | 0/409 [00:00<?, ?it/s]

Encode dev pool:   0%|          | 0/1023 [00:00<?, ?it/s]

Rank dev:   0%|          | 0/100 [00:00<?, ?it/s]

Dev retrieval: {'micro_precision_at_7': 0.015714285714285715, 'micro_recall_at_7': 0.014193548387096775, 'micro_f1_at_7': 0.014915254237288136, 'hits': 11, 'gold_relevant': 775, 'queries': 100, 'ranking_seconds': 34.480125931999964}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (31 outputs left out)

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Encode test pool:   0%|          | 0/1727 [00:00<?, ?it/s]

Rank test:   0%|          | 0/237 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path=WORK_DIR/'prediction.json'
with open(json_path,encoding='utf-8') as f:predictions=json.load(f)
validate_predictions(predictions,data['test'],candidate_pools['test'])
zip_path=WORK_DIR/SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:z.write(json_path,arcname='prediction.json')
print('Submission:',json_path,'| ZIP:',zip_path)

Submission: /kaggle/working/pcr_runs/pcr_rebuilt_encoder_decoder_mixed_lorma_len512/prediction.json | ZIP: /kaggle/working/pcr_runs/pcr_rebuilt_encoder_decoder_mixed_lorma_len512/pcr_rebuilt_encoder_decoder_mixed_lorma_len512.zip
